# Example 01.04 — run training

Starts the published AutoML pipeline and waits for completion. A successful run with
the same operation key is reused on later notebook executions.

Prerequisite: run notebooks 01.01–01.03.


In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "ml_app_client").is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Start Jupyter inside the ml-app repository")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from ml_app_client import MLAppClient, ResourceNotFoundError

client = MLAppClient.connect()
print("Connected to ML App")


## Choose resource names

These are normal names passed to `ml_app_client`. Edit them directly and use the
same values in the following notebooks. Dataset and pipeline names are resolved
inside the selected Business Case. Business Case and service names are globally
unique on one ML App installation.


In [ ]:
# These are ordinary platform resource names. Change the two globally unique
# names (Business Case and model service) when sharing one installation.
BUSINESS_CASE_NAME = "[MLAPP EXAMPLE 01 v2] Estates Lifecycle - demo"
TRAINING_DATASET_NAME = "Example01 Estates - Training"
SCORING_DATASET_NAME = "Example01 Estates - Batch Input"
ACTUALS_DATASET_NAME = "Example01 Estates - Actuals"
TRAINING_PIPELINE_NAME = "Example01 03 - AutoML Training"
BATCH_PIPELINE_NAME = "Example01 05 - Batch Scoring"
MONITORING_PIPELINE_NAME = "Example01 07 - Performance Monitoring"
MODEL_NAME = "Example01 Estates Price Model"
OUTPUT_NAME_PREFIX = "Example01 Estates AutoML"
MODEL_SERVICE_NAME = "Example01 10 - Estates Model Service - demo"

TRAINING_RUN_KEY = "Example01-training-v2"
BATCH_RUN_KEY = "Example01-batch-scoring-v2"
MONITORING_RUN_KEY = "Example01-monitoring-v2"

print("Resource names configured for:", BUSINESS_CASE_NAME)


In [ ]:
training = client.dataset_by_name(
    business_case_name=BUSINESS_CASE_NAME,
    dataset_name=TRAINING_DATASET_NAME,
)
try:
    run = client.pipeline_run_by_operation_key(business_case_name=BUSINESS_CASE_NAME, pipeline_name=TRAINING_PIPELINE_NAME, operation_key=TRAINING_RUN_KEY)
    started = False
except ResourceNotFoundError:
    run = client.run_pipeline_by_name(business_case_name=BUSINESS_CASE_NAME, pipeline_name=TRAINING_PIPELINE_NAME, runtime_parameters={"client_operation_key": TRAINING_RUN_KEY})
    started = True
print(f"{'STARTED' if started else 'REUSED'} run {run.id}; status={run.status}")

finished = client.wait_for_pipeline_run(
    run,
    timeout=3600,
    on_update=lambda current: print(
        f"status={current.status}; current_terminal_rows={current.processed_row_count}"
    ),
)
model = client.model_for_pipeline_run(finished)
if model["name"] != MODEL_NAME:
    raise RuntimeError(f"Expected model {MODEL_NAME!r}, got {model['name']!r}")
print({
    "run_id": finished.id,
    "full_training_scope_rows": training.row_count,
    "terminal_holdout_output_rows": finished.processed_row_count,
    "model_id": model["id"],
    "model_name": model["name"],
    "model_version": model["version"],
    "stage": model["stage"],
})
